# Generator-Verifier-Reviser パイプライン (v5)

## このノートブックの目的

LLM-jp Playground の 4 モデルを使って、研究タスク向け **Generator-Verifier-Reviser (GVR)**
パイプラインのプロトタイプを構築する。最終的には mdx クラスタ上の Claude/Gemini 主力パイプラインに
LLM-jp を「第3の意見」として組み込めるよう、**プロバイダ非依存の抽象化**を導入する。

## 構成

1. v4 ヘルパの取り込み (self-contained)
2. `ModelClient` — プロバイダ非依存の呼び出しラッパ
3. `VerifierResponse` + パーサ — 構造化出力の防御的解釈
4. `GVRPipeline` — Generator → Verifier → (Reviser → Verifier)* の orchestration
5. **Demo 1**: 数学命題の真偽判定 (易しい正命題 / 紛らわしい誤命題)
6. **Demo 2**: Julia コード生成 + レビュー
7. **Demo 3**: 多ラウンド改訂 (Chebyshev bias 関連のタスク)
8. マルチ Verifier 投票
9. トレースの保存・再読込
10. Claude / Gemini 統合のためのスタブ
11. mdx 並列ワークフローへの組み込み指針


## 1. セットアップ (v4 ヘルパを self-contained で再掲)

In [ ]:
import os, json, time, textwrap, random, re, datetime
from dataclasses import dataclass, field, asdict
from typing import Optional, Any, List, Dict, Callable, Tuple
from pathlib import Path
from openai import OpenAI, APIError, APIConnectionError, RateLimitError
from IPython.display import display, Markdown, HTML

BASE_URL = "https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1"
API_KEY = os.environ.get("LLMJP_API_KEY", "dummy")
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=300.0)

DEFAULT_MAX_TOKENS = 8000

# ----- LaTeX レンダリング -----
def latex_normalize(text):
    if not text:
        return text or ""
    text = re.sub(r'\\\[(.+?)\\\]', r'$$\1$$', text, flags=re.DOTALL)
    text = re.sub(r'\\\((.+?)\\\)', r'$\1$', text, flags=re.DOTALL)
    return text

def render_md(text):
    display(Markdown(latex_normalize(text or "_(empty)_")))


# ----- ChatResult + 内部 streaming -----
@dataclass
class ChatResult:
    text: str
    content: Optional[str]
    reasoning: Optional[str]
    finish_reason: Optional[str]
    usage: Optional[dict]
    model: str
    elapsed_sec: float

def _stream_collect(model, messages, **kwargs):
    kwargs.setdefault("max_tokens", DEFAULT_MAX_TOKENS)
    kwargs["stream"] = True
    kwargs.setdefault("stream_options", {"include_usage": True})
    t0 = time.time()
    stream = client.chat.completions.create(model=model, messages=messages, **kwargs)
    content_chunks, reasoning_chunks = [], []
    finish_reason, usage, actual_model = None, None, model
    for chunk in stream:
        if getattr(chunk, "model", None): actual_model = chunk.model
        if getattr(chunk, "usage", None):
            usage = chunk.usage.model_dump() if hasattr(chunk.usage, "model_dump") else dict(chunk.usage)
        if not chunk.choices: continue
        ch = chunk.choices[0]
        if ch.finish_reason: finish_reason = ch.finish_reason
        delta = ch.delta
        c = getattr(delta, "content", None)
        if c: content_chunks.append(c)
        for f in ("reasoning_content", "reasoning", "thinking"):
            v = getattr(delta, f, None)
            if v: reasoning_chunks.append(v); break
    dt = time.time() - t0
    content = "".join(content_chunks) if content_chunks else None
    reasoning = "".join(reasoning_chunks) if reasoning_chunks else None
    text = content or (f"[reasoning_only] {reasoning}" if reasoning else "(empty)")
    return ChatResult(text=text, content=content, reasoning=reasoning,
                      finish_reason=finish_reason, usage=usage,
                      model=actual_model, elapsed_sec=round(dt, 2))


def chat_once(model, user_msg, system=None, **kwargs):
    msgs = []
    if system: msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": user_msg})
    return _stream_collect(model, msgs, **kwargs)

def chat_once_nothink(model, user_msg, system=None, **kwargs):
    sys_msg = (system or "") + "\n\n/no_think"
    kwargs.setdefault("extra_body", {"chat_template_kwargs": {"enable_thinking": False}})
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user", "content": user_msg}]
    try:
        return _stream_collect(model, msgs, **kwargs)
    except Exception:
        kwargs.pop("extra_body", None)
        return _stream_collect(model, msgs, **kwargs)


# ----- モデルプローブ + smart_chat -----
PROBE_PROMPT = "1+1 を計算し、結果を一言で答えてください。"

def probe_model(model):
    info = {"model": model, "thinks": None, "nothink_works": None}
    try:
        r = chat_once(model, PROBE_PROMPT, system="日本語で。", max_tokens=2000, temperature=0.0)
        info["thinks"] = bool(r.reasoning)
    except Exception as e:
        info["err"] = repr(e)
        return info
    if info["thinks"]:
        try:
            r2 = chat_once_nothink(model, PROBE_PROMPT, system="日本語で。", max_tokens=1000, temperature=0.0)
            info["nothink_works"] = (not r2.reasoning) and bool(r2.content)
        except Exception:
            info["nothink_works"] = False
    return info

print("=== probing models... ===")
models = client.models.list()
model_ids = [m.id for m in models.data]
MODEL_BEHAVIOR = {}
for m in model_ids:
    info = probe_model(m)
    MODEL_BEHAVIOR[m] = info
    print(f"  {m}: thinks={info.get('thinks')} nothink_works={info.get('nothink_works')}")


def smart_chat(model, user_msg, system=None, disable_thinking=True, **kwargs):
    info = MODEL_BEHAVIOR.get(model, {})
    thinks = info.get("thinks", False)
    nothink_works = info.get("nothink_works", False)
    if not thinks or not disable_thinking:
        return chat_once(model, user_msg, system=system, **kwargs)
    if nothink_works:
        return chat_once_nothink(model, user_msg, system=system, **kwargs)
    kwargs.setdefault("max_tokens", 12000)
    return chat_once(model, user_msg, system=system, **kwargs)


# 代表モデル
def _pick(s):
    for m in model_ids:
        if s.lower() in m.lower(): return m
    return None

MODEL_LLMJP_SMALL = _pick("llm-jp-4-8b")
MODEL_LLMJP_LARGE = _pick("llm-jp-4-32b")
MODEL_QWEN        = _pick("qwen")
MODEL_GEMMA       = _pick("gemma")

print()
print("Models:")
print(f"  LLMJP_SMALL = {MODEL_LLMJP_SMALL}")
print(f"  LLMJP_LARGE = {MODEL_LLMJP_LARGE}")
print(f"  QWEN        = {MODEL_QWEN}")
print(f"  GEMMA       = {MODEL_GEMMA}")


## 2. `ModelClient` — プロバイダ非依存ラッパ

将来 Claude や Gemini を同じパイプラインに差し込めるよう、すべての LLM 呼び出しを
`ModelClient.__call__(prompt, system=None, **kwargs) -> ChatResult` インタフェースに統一する。


In [ ]:
@dataclass
class ModelClient:
    """プロバイダ非依存の呼び出しラッパ"""
    name: str
    call_fn: Callable[..., ChatResult]
    provider: str = "unknown"
    
    def __call__(self, prompt: str, system: Optional[str] = None, **kwargs) -> ChatResult:
        return self.call_fn(prompt, system=system, **kwargs)


def make_llmjp_client(model_id: str, force_thinking: Optional[bool] = None) -> ModelClient:
    """LLM-jp Playground 用クライアントを生成"""
    if force_thinking is True:
        fn = lambda p, system=None, **k: smart_chat(model_id, p, system=system, disable_thinking=False, **k)
    elif force_thinking is False:
        fn = lambda p, system=None, **k: smart_chat(model_id, p, system=system, disable_thinking=True, **k)
    else:
        fn = lambda p, system=None, **k: smart_chat(model_id, p, system=system, **k)
    return ModelClient(name=model_id, call_fn=fn, provider="llmjp-playground")


# 主要クライアントを用意
C_LLMJP_8B  = make_llmjp_client(MODEL_LLMJP_SMALL)
C_LLMJP_32B = make_llmjp_client(MODEL_LLMJP_LARGE)
C_QWEN      = make_llmjp_client(MODEL_QWEN)             # デフォルト: thinking 抑制
C_QWEN_TH   = make_llmjp_client(MODEL_QWEN, force_thinking=True)  # thinking ON
C_GEMMA     = make_llmjp_client(MODEL_GEMMA)
C_GEMMA_TH  = make_llmjp_client(MODEL_GEMMA, force_thinking=True)

ALL_CLIENTS = [C_LLMJP_8B, C_LLMJP_32B, C_QWEN, C_GEMMA]
print("Clients ready:", [c.name for c in ALL_CLIENTS])


## 3. Verifier 出力の構造化

Verifier には決まった形式で返してもらうよう指示し、防御的にパースする。
JSON 強制はモデル次第で失敗しやすいので、**ラベル付きセクション形式**で出してもらう。


In [ ]:
VERIFIER_TEMPLATE = textwrap.dedent("""\
    あなたは厳格な数学・コードレビュアーです。以下の主張または成果物を評価してください。
    
    [TASK]
    {task}
    [/TASK]
    
    [SUBMISSION]
    {submission}
    [/SUBMISSION]
    
    以下の形式で**厳密に**回答してください。各セクションは必ずキーで始めること。
    
    VERDICT: CORRECT | PARTIAL | INCORRECT
    CONFIDENCE: HIGH | MEDIUM | LOW
    REASONS:
    - (主張の正誤を支える理由を箇条書き、各行ハイフン始まり)
    ISSUES:
    - (見つかった問題点。なければ「なし」)
    SUGGESTIONS:
    - (改訂の提案。なければ「なし」)
""")


@dataclass
class VerifierResponse:
    verdict: str        # CORRECT / PARTIAL / INCORRECT / UNKNOWN
    confidence: str     # HIGH / MEDIUM / LOW / UNKNOWN
    reasons: List[str]
    issues: List[str]
    suggestions: List[str]
    raw: str            # 元の本文
    parse_ok: bool


_VERDICT_PAT = re.compile(r'VERDICT\s*[:：]\s*(CORRECT|PARTIAL|INCORRECT)', re.I)
_CONF_PAT    = re.compile(r'CONFIDENCE\s*[:：]\s*(HIGH|MEDIUM|LOW)', re.I)
_SECTION_PAT = re.compile(r'^\s*(REASONS|ISSUES|SUGGESTIONS)\s*[:：]\s*$', re.I)


def parse_verifier(text: str) -> VerifierResponse:
    text = text or ""
    verdict_m = _VERDICT_PAT.search(text)
    conf_m = _CONF_PAT.search(text)
    verdict = verdict_m.group(1).upper() if verdict_m else "UNKNOWN"
    confidence = conf_m.group(1).upper() if conf_m else "UNKNOWN"

    sections = {"REASONS": [], "ISSUES": [], "SUGGESTIONS": []}
    current = None
    for line in text.splitlines():
        m = _SECTION_PAT.match(line)
        if m:
            current = m.group(1).upper()
            continue
        # セクションが終わりそうな別キーが来たら抜ける
        if re.match(r'^\s*(VERDICT|CONFIDENCE)\s*[:：]', line, re.I):
            current = None
            continue
        if current:
            stripped = line.strip()
            if stripped.startswith(("-", "・", "*")):
                item = stripped.lstrip("-・* ").strip()
                if item and item != "なし":
                    sections[current].append(item)

    parse_ok = (verdict != "UNKNOWN")
    return VerifierResponse(
        verdict=verdict,
        confidence=confidence,
        reasons=sections["REASONS"],
        issues=sections["ISSUES"],
        suggestions=sections["SUGGESTIONS"],
        raw=text,
        parse_ok=parse_ok,
    )


# パーサ自体の動作確認
sample = textwrap.dedent("""\
    VERDICT: PARTIAL
    CONFIDENCE: MEDIUM
    REASONS:
    - 主張の前半は正しい
    - ただし境界条件が抜けている
    ISSUES:
    - n=0 の場合の扱いが不明
    SUGGESTIONS:
    - n>=1 と明記すること
""")
vr = parse_verifier(sample)
print(vr)
assert vr.verdict == "PARTIAL"
assert vr.confidence == "MEDIUM"
assert len(vr.reasons) == 2
assert len(vr.issues) == 1
assert len(vr.suggestions) == 1
print("\nparser OK ✓")


## 4. `GVRPipeline` 本体

Generator → Verifier → (Reviser → Verifier)* のループ。最大 `max_revisions` 回まで改訂。
すべての中間結果は `PipelineTrace` に記録され、後で可視化・保存可能。


In [ ]:
GENERATOR_DEFAULT_SYSTEM = "あなたは厳密な数学・科学アシスタントです。日本語で、必要なら LaTeX を使って答えてください。"

REVISER_TEMPLATE = textwrap.dedent("""\
    元のタスクとあなたの前回の回答、それに対する Verifier の指摘を踏まえて、
    改訂版の回答を作成してください。
    
    [TASK]
    {task}
    [/TASK]
    
    [PREVIOUS_ANSWER]
    {previous}
    [/PREVIOUS_ANSWER]
    
    [VERIFIER_FEEDBACK]
    判定: {verdict} (信頼度: {confidence})
    問題点:
    {issues}
    
    改訂提案:
    {suggestions}
    [/VERIFIER_FEEDBACK]
    
    上記を取り入れて、改訂版の回答だけを出力してください (メタコメントは不要)。
""")


@dataclass
class PipelineStep:
    role: str              # "generator" | "verifier" | "reviser"
    model: str
    iteration: int
    prompt: str
    output: Optional[str]
    reasoning: Optional[str]
    parsed: Optional[VerifierResponse] = None
    finish_reason: Optional[str] = None
    usage: Optional[dict] = None
    elapsed_sec: float = 0.0


@dataclass
class PipelineTrace:
    task: str
    steps: List[PipelineStep] = field(default_factory=list)
    final_output: Optional[str] = None
    final_verdict: Optional[str] = None
    success: bool = False
    iterations: int = 0
    total_elapsed_sec: float = 0.0
    started_at: Optional[str] = None
    
    def to_dict(self) -> dict:
        d = asdict(self)
        # VerifierResponse も dict 化される
        return d


def _format_list(items: List[str]) -> str:
    return "\n".join(f"- {x}" for x in items) if items else "- なし"


class GVRPipeline:
    def __init__(self,
                 generator: ModelClient,
                 verifier: ModelClient,
                 reviser: Optional[ModelClient] = None,
                 max_revisions: int = 2,
                 generator_system: str = GENERATOR_DEFAULT_SYSTEM,
                 verifier_template: str = VERIFIER_TEMPLATE,
                 reviser_template: str = REVISER_TEMPLATE,
                 success_verdicts: Tuple[str, ...] = ("CORRECT",),
                 stop_on_verifier_failure: bool = False):
        self.generator = generator
        self.verifier = verifier
        self.reviser = reviser
        self.max_revisions = max_revisions
        self.generator_system = generator_system
        self.verifier_template = verifier_template
        self.reviser_template = reviser_template
        self.success_verdicts = success_verdicts
        self.stop_on_verifier_failure = stop_on_verifier_failure
    
    def _call_generator(self, task: str, iteration: int, **kwargs) -> PipelineStep:
        r = self.generator(task, system=self.generator_system, **kwargs)
        return PipelineStep(
            role="generator", model=self.generator.name, iteration=iteration,
            prompt=task, output=r.content, reasoning=r.reasoning,
            finish_reason=r.finish_reason, usage=r.usage, elapsed_sec=r.elapsed_sec,
        )
    
    def _call_verifier(self, task: str, submission: str, iteration: int, **kwargs) -> PipelineStep:
        prompt = self.verifier_template.format(task=task, submission=submission)
        r = self.verifier(prompt, **kwargs)
        parsed = parse_verifier(r.content or r.reasoning or "")
        return PipelineStep(
            role="verifier", model=self.verifier.name, iteration=iteration,
            prompt=prompt, output=r.content, reasoning=r.reasoning,
            parsed=parsed, finish_reason=r.finish_reason, usage=r.usage,
            elapsed_sec=r.elapsed_sec,
        )
    
    def _call_reviser(self, task: str, previous: str, vr: VerifierResponse,
                      iteration: int, **kwargs) -> PipelineStep:
        prompt = self.reviser_template.format(
            task=task,
            previous=previous,
            verdict=vr.verdict,
            confidence=vr.confidence,
            issues=_format_list(vr.issues),
            suggestions=_format_list(vr.suggestions),
        )
        client = self.reviser or self.generator   # reviser がなければ generator 自身で再書き
        r = client(prompt, system=self.generator_system, **kwargs)
        return PipelineStep(
            role="reviser", model=client.name, iteration=iteration,
            prompt=prompt, output=r.content, reasoning=r.reasoning,
            finish_reason=r.finish_reason, usage=r.usage, elapsed_sec=r.elapsed_sec,
        )
    
    def run(self, task: str, **kwargs) -> PipelineTrace:
        trace = PipelineTrace(task=task,
                              started_at=datetime.datetime.now().isoformat())
        t0 = time.time()
        # 1. Generate
        gen_step = self._call_generator(task, iteration=0, **kwargs)
        trace.steps.append(gen_step)
        current_answer = gen_step.output or ""
        
        # 2. Verify
        ver_step = self._call_verifier(task, current_answer, iteration=0, **kwargs)
        trace.steps.append(ver_step)
        
        # 3. Revise loop
        for it in range(1, self.max_revisions + 1):
            vr = ver_step.parsed
            if vr and vr.verdict in self.success_verdicts:
                break
            if self.stop_on_verifier_failure and not (vr and vr.parse_ok):
                break
            # revise
            rev_step = self._call_reviser(task, current_answer, vr,
                                          iteration=it, **kwargs)
            trace.steps.append(rev_step)
            current_answer = rev_step.output or current_answer
            # re-verify
            ver_step = self._call_verifier(task, current_answer, iteration=it, **kwargs)
            trace.steps.append(ver_step)
        
        trace.final_output = current_answer
        trace.final_verdict = ver_step.parsed.verdict if ver_step.parsed else "UNKNOWN"
        trace.success = trace.final_verdict in self.success_verdicts
        trace.iterations = max((s.iteration for s in trace.steps), default=0)
        trace.total_elapsed_sec = round(time.time() - t0, 2)
        return trace


## 5. トレース可視化


In [ ]:
ROLE_EMOJI = {"generator": "🟦", "verifier": "🟨", "reviser": "🟧"}
VERDICT_EMOJI = {"CORRECT": "✅", "PARTIAL": "⚠️", "INCORRECT": "❌", "UNKNOWN": "❓"}


def render_trace(trace: PipelineTrace, show_reasoning_head: int = 0):
    """PipelineTrace を見やすく Markdown 表示"""
    head = (f"# {VERDICT_EMOJI.get(trace.final_verdict, '❓')} "
            f"GVR Pipeline: `{trace.final_verdict}` "
            f"({'success' if trace.success else 'failed'})")
    summary = (f"**iterations**: {trace.iterations} · "
               f"**elapsed**: {trace.total_elapsed_sec}s · "
               f"**steps**: {len(trace.steps)}")
    display(Markdown(f"{head}\n\n{summary}\n\n---\n\n### 🟦 Task\n\n{latex_normalize(trace.task)}\n\n---"))
    
    for s in trace.steps:
        emoji = ROLE_EMOJI.get(s.role, "•")
        title = f"### {emoji} Step {s.iteration}.{s.role} — `{s.model}`"
        meta = f"`elapsed={s.elapsed_sec}s` · `finish={s.finish_reason}`"
        body_parts = [title, "", meta, ""]
        
        if s.role == "verifier" and s.parsed:
            v = s.parsed
            body_parts.append(
                f"**Verdict**: {VERDICT_EMOJI.get(v.verdict,'❓')} `{v.verdict}` "
                f"· **Confidence**: `{v.confidence}` · **Parsed**: `{v.parse_ok}`"
            )
            if v.reasons:
                body_parts.append("\n**Reasons:**\n" + "\n".join(f"- {latex_normalize(x)}" for x in v.reasons))
            if v.issues:
                body_parts.append("\n**Issues:**\n" + "\n".join(f"- {latex_normalize(x)}" for x in v.issues))
            if v.suggestions:
                body_parts.append("\n**Suggestions:**\n" + "\n".join(f"- {latex_normalize(x)}" for x in v.suggestions))
        else:
            body = s.output or (f"_(reasoning_only)_\n\n{s.reasoning}" if s.reasoning else "_(empty)_")
            body_parts.append(latex_normalize(body))
        
        if show_reasoning_head and s.reasoning:
            body_parts.append(f"\n<details><summary>reasoning (head {show_reasoning_head}c)</summary>\n\n```\n"
                              + s.reasoning[:show_reasoning_head] + "\n```\n</details>")
        body_parts.append("\n---")
        display(Markdown("\n".join(body_parts)))


## 6. Demo 1: 数学命題の真偽判定

簡単な事例で動作確認。
- **正命題**: 「100 以下の素数で $p \equiv 1 \pmod 4$ となるものは 24 個ある」
- **誤命題**: 「フェルマー素数 $F_n = 2^{2^n}+1$ は全て素数である」(古典的な誤り — F_5 が反例)


In [ ]:
# パイプライン: Generator=LLM-jp 32b, Verifier=LLM-jp 8b, Reviser=LLM-jp 32b
pipe_simple = GVRPipeline(
    generator=C_LLMJP_32B,
    verifier=C_LLMJP_8B,
    reviser=C_LLMJP_32B,
    max_revisions=1,
)

task1 = "100 以下の素数で $p \\equiv 1 \\pmod 4$ となるものをすべて列挙し、その個数を答えよ。"
trace1 = pipe_simple.run(task1, temperature=0.0)
render_trace(trace1)


In [ ]:
task2 = textwrap.dedent("""\
    次の命題の真偽を判定し、根拠を示しなさい。
    
    『$F_n = 2^{2^n} + 1$ の形のフェルマー数はすべて素数である。』
""").strip()

trace2 = pipe_simple.run(task2, temperature=0.0)
render_trace(trace2)


## 7. Demo 2: コード生成 + レビュー

Julia コードを Generator に書かせ、別モデルで Verifier する。Verifier の Reasoning が
活きる場面 — Qwen3.6 を thinking ON でレビュアーに据えるとどうなるか。


In [ ]:
# Generator: LLM-jp 32b (日本語コメント付きコードに向く)
# Verifier:  Qwen3.6 thinking ON (慎重なコードレビュー)
# Reviser:   LLM-jp 32b
pipe_code = GVRPipeline(
    generator=C_LLMJP_32B,
    verifier=C_QWEN_TH,
    reviser=C_LLMJP_32B,
    max_revisions=2,
    success_verdicts=("CORRECT",),
)

code_task = textwrap.dedent("""\
    Julia で次の関数を書いてください:
    
    `chebyshev_imbalance(N::Int, q::Int) :: Tuple{Int,Int}`
    
    入力:
      - N: 上限の自然数
      - q: 法 (q >= 3 を想定)
    
    動作: 2 から N までの素数 p を列挙し、
      - π_1(N): p ≡ 1 (mod q) を満たす素数の個数
      - π_3(N): p ≡ q-1 (mod q) を満たす素数の個数 (q が偶数なら q+1 でも可、ここでは -1 ≡ q-1 を採用)
    を計算して (π_1, π_3) を返してください。
    
    コードのみ出力し、簡潔なコメントを付けてください。
""").strip()

trace_code = pipe_code.run(code_task, temperature=0.0)
render_trace(trace_code, show_reasoning_head=400)


## 8. Demo 3: 多ラウンド改訂 (Chebyshev bias 関連タスク)

研究文脈に近い、論述系タスクで改訂ループを観察する。


In [ ]:
pipe_research = GVRPipeline(
    generator=C_LLMJP_32B,
    verifier=C_GEMMA_TH,        # Gemma を thinking ON でレビュアーに
    reviser=C_LLMJP_32B,
    max_revisions=2,
    # PARTIAL も成功扱いにして打ち切り条件を緩める
    success_verdicts=("CORRECT", "PARTIAL"),
)

research_task = textwrap.dedent("""\
    円分体 $\\mathbb{Q}(\\zeta_p)$ ($p$ 奇素数) の相対類数 $h^-_p$ について、
    岩澤理論の $\\lambda$-不変量 $\\lambda_p$ と、Bernoulli 数の一般化である一般化ベルヌーイ数
    $B_{1,\\chi}$ の $p$-進付値との関係を、専門家向けに 1 段落で説明してください。
    
    重要: 厳密な数学的事実のみを述べ、推測や曖昧な表現は避けてください。
    関連する経験的観察 (例えば $\\sum_\\chi v_p(B_{1,\\chi}) = \\lambda_p - 1$ のような) があれば、
    それが定理なのか経験則なのかを明確に区別してください。
""").strip()

trace_research = pipe_research.run(research_task, temperature=0.2)
render_trace(trace_research, show_reasoning_head=300)


## 9. マルチ Verifier 投票

1人のVerifierでは判定が不安定なことがあるので、複数モデルで投票する。


In [ ]:
def multi_verify(task: str, submission: str,
                 verifiers: List[ModelClient],
                 **kwargs) -> Dict[str, Any]:
    """複数 verifier に同じ判定をさせて投票"""
    prompt = VERIFIER_TEMPLATE.format(task=task, submission=submission)
    individual = []
    for v in verifiers:
        r = v(prompt, **kwargs)
        parsed = parse_verifier(r.content or r.reasoning or "")
        individual.append({
            "verifier": v.name,
            "verdict": parsed.verdict,
            "confidence": parsed.confidence,
            "issues": parsed.issues,
            "elapsed": r.elapsed_sec,
        })
    
    # 単純多数決
    from collections import Counter
    counts = Counter(x["verdict"] for x in individual)
    majority = counts.most_common(1)[0]
    return {
        "majority_verdict": majority[0],
        "majority_count": majority[1],
        "total": len(verifiers),
        "agreement_ratio": majority[1] / len(verifiers),
        "individual": individual,
    }


# Demo 1 task2 (フェルマー素数命題) の Generator 出力を取り出して再投票
gen_output = next(s.output for s in trace2.steps if s.role == "generator")
print("Generator output (head 200c):", (gen_output or "")[:200])
print()
print("=== multi-verifier vote ===")
mv = multi_verify(task2, gen_output or "",
                  verifiers=[C_LLMJP_8B, C_LLMJP_32B, C_QWEN_TH, C_GEMMA_TH],
                  temperature=0.0)

display(Markdown(f"""
### Multi-Verifier Vote
- **Majority**: {VERDICT_EMOJI.get(mv['majority_verdict'],'❓')} `{mv['majority_verdict']}` ({mv['majority_count']}/{mv['total']})
- **Agreement**: {mv['agreement_ratio']*100:.0f}%

| Verifier | Verdict | Confidence | Issues |
|---|---|---|---|
""" + "\n".join(
    f"| `{x['verifier']}` | {VERDICT_EMOJI.get(x['verdict'],'❓')} {x['verdict']} | {x['confidence']} | {len(x['issues'])} |"
    for x in mv['individual']
)))


## 10. トレースの保存・再読込

JSON で永続化して、後で別ノートブックや解析スクリプトから読める形式に。


In [ ]:
TRACE_DIR = Path("./gvr_traces")
TRACE_DIR.mkdir(exist_ok=True)


def save_trace(trace: PipelineTrace, filename: Optional[str] = None) -> Path:
    if filename is None:
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        gen = trace.steps[0].model if trace.steps else "unknown"
        filename = f"trace_{stamp}_{gen.replace('/','_')}.json"
    path = TRACE_DIR / filename
    with path.open("w", encoding="utf-8") as f:
        json.dump(trace.to_dict(), f, ensure_ascii=False, indent=2)
    print(f"saved: {path} ({path.stat().st_size} bytes)")
    return path


def load_trace(path) -> dict:
    """dict として読み込む (PipelineTrace への再構築はせず軽量に保つ)"""
    return json.loads(Path(path).read_text(encoding="utf-8"))


# 保存
for tag, tr in [("simple1", trace1), ("simple2_fermat", trace2),
                ("code", trace_code), ("research", trace_research)]:
    p = save_trace(tr, filename=f"demo_{tag}.json")

# 読込確認
loaded = load_trace(TRACE_DIR / "demo_simple2_fermat.json")
print()
print("loaded keys:", list(loaded.keys()))
print("step count :", len(loaded["steps"]))
print("final verdict:", loaded["final_verdict"])


## 11. Claude / Gemini 統合のスタブ

`ModelClient` インタフェースに合わせれば、他プロバイダも同じパイプラインに差し込める。
以下は **未テスト** だが構造は動くはず — 横山さんの環境に `ANTHROPIC_API_KEY` / `GEMINI_API_KEY`
が設定されていれば、コメントアウトを外して使えるはず。


In [ ]:
# --- Claude (Anthropic) スタブ ---
def make_claude_client(model: str = "claude-opus-4-7") -> ModelClient:
    """Anthropic SDK 経由の ModelClient (要 anthropic パッケージ + API キー)"""
    try:
        from anthropic import Anthropic
    except ImportError:
        print("anthropic package not installed; pip install anthropic")
        return None
    ac = Anthropic()  # ANTHROPIC_API_KEY を環境変数から
    
    def call_fn(prompt, system=None, **kwargs):
        t0 = time.time()
        params = dict(
            model=model,
            max_tokens=kwargs.get("max_tokens", 4096),
            messages=[{"role": "user", "content": prompt}],
        )
        if system:
            params["system"] = system
        if "temperature" in kwargs:
            params["temperature"] = kwargs["temperature"]
        resp = ac.messages.create(**params)
        dt = time.time() - t0
        content = "".join(b.text for b in resp.content if hasattr(b, "text"))
        return ChatResult(
            text=content, content=content, reasoning=None,
            finish_reason=resp.stop_reason,
            usage={"input_tokens": resp.usage.input_tokens,
                   "output_tokens": resp.usage.output_tokens},
            model=resp.model, elapsed_sec=round(dt, 2),
        )
    return ModelClient(name=f"claude:{model}", call_fn=call_fn, provider="anthropic")


# --- Gemini (Google) スタブ ---
def make_gemini_client(model: str = "gemini-2.5-pro") -> ModelClient:
    try:
        from google import genai
    except ImportError:
        print("google-genai package not installed; pip install google-genai")
        return None
    gc = genai.Client()  # GOOGLE_API_KEY / GEMINI_API_KEY を環境変数から
    
    def call_fn(prompt, system=None, **kwargs):
        t0 = time.time()
        full_prompt = (f"{system}\n\n{prompt}" if system else prompt)
        resp = gc.models.generate_content(model=model, contents=full_prompt)
        dt = time.time() - t0
        return ChatResult(
            text=resp.text, content=resp.text, reasoning=None,
            finish_reason="stop", usage=None,
            model=model, elapsed_sec=round(dt, 2),
        )
    return ModelClient(name=f"gemini:{model}", call_fn=call_fn, provider="google")


# 使用例 (キー設定後にコメントアウトを外す)
# C_CLAUDE = make_claude_client("claude-opus-4-7")
# C_GEMINI = make_gemini_client("gemini-2.5-pro")
#
# pipe_heterogeneous = GVRPipeline(
#     generator=C_CLAUDE,         # Claude が生成
#     verifier=C_GEMINI,          # Gemini が検証
#     reviser=C_LLMJP_32B,        # LLM-jp が改訂 (日本語の自然さ)
#     max_revisions=2,
# )

print("Claude/Gemini stubs ready (commented out — set API keys and uncomment to use)")


## 12. mdx 並列ワークフローへの組み込み指針

横山さんの既存ワークフロー (5 Pane Claude Code CLI + mdx-01〜05) に組み込む場合の指針:

### 役割分担の案

| 役割 | おすすめ | 理由 |
|---|---|---|
| **Generator** (主) | Claude Opus 4.7 (Pane 1) | 数学厳密性とコード品質 |
| **Generator** (副) | LLM-jp-4-32b-a3b | 日本語論述の自然さ、コスト無料 |
| **Verifier** (主) | Gemini 2.5 Pro | Claude と異なる視点 |
| **Verifier** (副) | Qwen3.6-27B thinking ON | 第3の独立視点、コスト無料 |
| **Reviser** | Claude Opus 4.7 | 指摘を踏まえた書き直しの質 |

### 実装パターン

```python
# 1. ハンドオフ時に GVRPipeline を構築
pipe = GVRPipeline(
    generator=make_claude_client("claude-opus-4-7"),
    verifier=make_gemini_client("gemini-2.5-pro"),
    reviser=make_claude_client("claude-opus-4-7"),
    max_revisions=2,
)

# 2. 各 Pane が異なる Generator で並列に試行
# Pane 1: Claude
# Pane 2: LLM-jp 32B
# Pane 3: Gemini
# → 3 trace を集約して比較

# 3. node 上の計算結果検証にも応用可能
#   - Generator: 「この計算結果は λ-不変量と整合するか?」
#   - Verifier: 別モデルで独立検証
```

### コスト試算 (1タスク = Gen + Ver + 最大2回 Rev + Ver = 最大6呼び出し)

- Claude Opus 4.7: ~6000 input + 2000 output tokens × 4 calls = ~$0.5/task
- Gemini 2.5 Pro: ~6000 input + 1500 output × 2 calls = ~$0.05/task
- LLM-jp Playground: **無料** (NII 内部)

**結論**: LLM-jp を Verifier / Reviser の副プールに据えると、月数百タスクの試行が現実的になる。

### 次のステップ

1. **このノートブックを mdx-01 で実行** して BASE_URL に到達可能か検証
2. `gvr_traces/` を SINETStream / Gakunin RDM に同期して履歴を蓄積
3. Aoki 先生レビュー前のセルフチェックとして `pipe_research` 形式を組み込む
4. Paper B のセクション草稿に対する自動レビュー (Verifier 多数決) を試す


In [ ]:
# 全 trace のサマリ
print("=== すべての trace サマリ ===")
for path in sorted(TRACE_DIR.glob("demo_*.json")):
    d = load_trace(path)
    print(f"{path.name}: verdict={d['final_verdict']}, "
          f"iters={d['iterations']}, "
          f"steps={len(d['steps'])}, "
          f"elapsed={d['total_elapsed_sec']}s")


## まとめ

**v5 で確立された資産**

1. **`ModelClient`**: プロバイダ非依存の呼び出しラッパ → Claude/Gemini/LLM-jp を統一インタフェースで扱える
2. **`VerifierResponse` パーサ**: 構造化出力を防御的に解釈、JSON 強制よりロバスト
3. **`GVRPipeline`**: 4-step 〜 6-step のループ orchestration、`max_revisions` で打ち切り制御
4. **`PipelineTrace`**: 全中間結果の構造化記録 → JSON 永続化
5. **`render_trace`**: Markdown + LaTeX で美しく可視化
6. **マルチ Verifier 投票**: 単一判定の不安定性を回避
7. **Claude / Gemini スタブ**: 既存パイプラインへの差し込み準備完了

**未対応・次の改善余地**

- **非同期化**: 現状は逐次呼び出し。`asyncio` + Anthropic/OpenAI の async client で並列化すると、
  4-verifier 投票が 4x 早くなる
- **プロンプトキャッシング**: Verifier テンプレートは固定なので、Anthropic prompt caching で
  入力コストを削減可能
- **Lean 4 連携**: Verifier が "INCORRECT" を返したら、対応する Lean 4 検証スクリプトを自動生成 →
  Mathlib で機械的に検証 — 横山さんの「Phase 0/1/2 計画」と整合
- **トレース DB**: SQLite に集約して、月単位で「どのモデルが最も信頼できる Verifier だったか」を集計
